<a href="https://colab.research.google.com/github/subudear/deep-learning/blob/main/assignment1/GAN/dcgan_dinov2_petridish.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DC-GAN + DINOv2-ViT-B/14 — Petri Dish Cell Classification

## How DC-GAN solves class imbalance

```
Real minority-class images
        │
        ▼
┌───────────────────┐     noise z ~ N(0,I)
│  GENERATOR (G)    │◄────────────────────
│  ConvTranspose    │◄──── class label c
│  blocks           │
└───────┬───────────┘
        │ fake 64x64 images (upscaled to 448 for DINOv2)
        ▼
┌───────────────────┐◄──── real images
│  DISCRIMINATOR(D) │◄──── class label c
│  Conv blocks      │
└───────┬───────────┘
        │ BCE loss
        ▼
    G fools D, D detects fakes → adversarial equilibrium
```

**Full pipeline:**
1. Conditional DC-GAN trains on minority-class images → synthesises class-conditioned images
2. GAN augmentation brings every minority class to GAN_TARGET (val set never touched)
3. DINOv2-ViT-B/14 classifies the balanced dataset using 3-phase progressive fine-tuning
4. Focal Loss (γ=2) focuses training on hard/rare examples

*DC-GAN Reference: Ma et al. "Combining DC-GAN with ResNet for blood cell image classification", MBE&C, 2020*

## STEP 1 — Imports & GPU check

In [ ]:
import os
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name:', torch.cuda.get_device_name(0))
    print('GPU memory:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

## STEP 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

## STEP 3 — Extract dataset

In [ ]:
dataset_path = '/content/drive/MyDrive/compx525assign1/cell_cultures'
extract_path = '/content/dataset'
os.makedirs(extract_path, exist_ok=True)
for f in os.listdir(dataset_path):
    if f.endswith('.zip'):
        zip_path = os.path.join(dataset_path, f)
        print('Extracting:', zip_path)
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(extract_path)
print('Done. Folders:', os.listdir(extract_path))

## STEP 4 — Define paths & audit dataset

In [ ]:
train_dir  = '/content/dataset/train'
test_dir   = '/content/dataset/test'
image_exts = ('.jpg', '.jpeg', '.png')

print('=== DATASET AUDIT ===')
class_counts = {}
for cls in sorted(os.listdir(train_dir)):
    cls_path = os.path.join(train_dir, cls)
    if not os.path.isdir(cls_path):
        continue
    n = sum(f.lower().endswith(image_exts) for f in os.listdir(cls_path))
    class_counts[cls] = n
    tier = 'EMPTY' if n == 0 else 'EXTREME(<15)' if n < 15 else 'MINORITY(<100)' if n < 100 else 'OK'
    print(f'  {cls}: {n:5d}  [{tier}]')

test_imgs = [f for f in os.listdir(test_dir) if f.lower().endswith(image_exts)]
print(f'\nTotal classes: {len(class_counts)}')
print(f'Total train images: {sum(class_counts.values())}')
print(f'Total test images:  {len(test_imgs)}')

EMPTY_CLASSES    = [c for c, n in class_counts.items() if n == 0]
EXTREME_CLASSES  = [c for c, n in class_counts.items() if 0 < n < 15]
GAN_CLASSES      = [c for c, n in class_counts.items() if 0 < n < 100]
print(f'\nEmpty (excluded):       {EMPTY_CLASSES}')
print(f'Extreme minority (<15): {EXTREME_CLASSES}')
print(f'GAN augment targets:    {GAN_CLASSES}')

## STEP 5 — Build training dataframe, train/val split & label encoding

In [ ]:
rows = []
for cls, n in class_counts.items():
    if cls in EMPTY_CLASSES:
        continue
    cls_path = os.path.join(train_dir, cls)
    for fname in os.listdir(cls_path):
        if fname.lower().endswith(image_exts):
            rows.append({'filepath': os.path.join(cls_path, fname), 'label': cls})

train_df = pd.DataFrame(rows)
print(f'Total usable training images: {len(train_df)}')

extreme_df    = train_df[train_df['label'].isin(EXTREME_CLASSES)].copy()
splittable_df = train_df[~train_df['label'].isin(EXTREME_CLASSES)].copy()
train_split_df, val_split_df = train_test_split(
    splittable_df, test_size=0.2, stratify=splittable_df['label'], random_state=42)
train_split_df = pd.concat([train_split_df, extreme_df], ignore_index=True)

all_labels   = sorted(train_df['label'].unique())
label_to_idx = {label: idx for idx, label in enumerate(all_labels)}
idx_to_label = {idx: label for label, idx in label_to_idx.items()}
num_classes  = len(label_to_idx)

train_split_df = train_split_df.copy()
val_split_df   = val_split_df.copy()
train_split_df['label_idx'] = train_split_df['label'].map(label_to_idx)
val_split_df['label_idx']   = val_split_df['label'].map(label_to_idx)

print(f'Train: {len(train_split_df)}  |  Val: {len(val_split_df)}  |  Classes: {num_classes}')

## STEP 6 — Define Conditional DC-GAN

In [ ]:
GAN_IMG_SIZE   = 64
GAN_NOISE_DIM  = 100
GAN_LABEL_EMB  = 50
GAN_EPOCHS     = 200
GAN_BATCH_SIZE = 64
GAN_LR         = 2e-4
GAN_TARGET     = 200


class Generator(nn.Module):
    def __init__(self, noise_dim, num_classes, label_emb_dim):
        super().__init__()
        self.label_emb = nn.Embedding(num_classes, label_emb_dim)
        in_dim = noise_dim + label_emb_dim
        self.net = nn.Sequential(
            nn.Linear(in_dim, 512 * 4 * 4),
            nn.Unflatten(1, (512, 4, 4)),
            nn.ConvTranspose2d(512, 256, 4, 2, 1, bias=False), nn.BatchNorm2d(256), nn.ReLU(True),
            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False), nn.BatchNorm2d(128), nn.ReLU(True),
            nn.ConvTranspose2d(128,  64, 4, 2, 1, bias=False), nn.BatchNorm2d(64),  nn.ReLU(True),
            nn.ConvTranspose2d( 64,   3, 4, 2, 1, bias=False), nn.Tanh(),
        )
    def forward(self, noise, labels):
        return self.net(torch.cat([noise, self.label_emb(labels)], dim=1))


class Discriminator(nn.Module):
    def __init__(self, num_classes, label_emb_dim, img_size=64):
        super().__init__()
        self.img_size   = img_size
        self.label_emb  = nn.Embedding(num_classes, label_emb_dim)
        self.label_proj = nn.Linear(label_emb_dim, img_size * img_size)
        self.net = nn.Sequential(
            nn.Conv2d(4,   64,  4, 2, 1, bias=False), nn.LeakyReLU(0.2, True),
            nn.Conv2d(64,  128, 4, 2, 1, bias=False), nn.BatchNorm2d(128), nn.LeakyReLU(0.2, True),
            nn.Conv2d(128, 256, 4, 2, 1, bias=False), nn.BatchNorm2d(256), nn.LeakyReLU(0.2, True),
            nn.Conv2d(256, 512, 4, 2, 1, bias=False), nn.BatchNorm2d(512), nn.LeakyReLU(0.2, True),
            nn.Conv2d(512,   1, 4, 1, 0, bias=False), nn.Sigmoid(),
        )
    def forward(self, img, labels):
        lmap = self.label_proj(self.label_emb(labels)).view(-1, 1, self.img_size, self.img_size)
        return self.net(torch.cat([img, lmap], dim=1)).view(-1)


def weights_init(m):
    name = m.__class__.__name__
    if 'Conv' in name or 'Linear' in name:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif 'BatchNorm' in name:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

print('DC-GAN architecture defined.')

## STEP 7 — Train DC-GAN on minority classes

In [ ]:
gan_class_list   = sorted(GAN_CLASSES)
gan_label_to_idx = {c: i for i, c in enumerate(gan_class_list)}
num_gan_classes  = len(gan_class_list)

gan_rows = [r for r in rows if r['label'] in gan_class_list]
gan_df   = pd.DataFrame(gan_rows)
gan_df['label_idx'] = gan_df['label'].map(gan_label_to_idx)
print(f'GAN training: {num_gan_classes} minority classes, {len(gan_df)} real images')

gan_transform = transforms.Compose([
    transforms.Resize((GAN_IMG_SIZE, GAN_IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

class GANDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return self.transform(Image.open(row['filepath']).convert('RGB')), int(row['label_idx'])

gan_loader = DataLoader(GANDataset(gan_df, gan_transform),
                        batch_size=GAN_BATCH_SIZE, shuffle=True,
                        num_workers=2, pin_memory=True, drop_last=True)

netG = Generator(GAN_NOISE_DIM, num_gan_classes, GAN_LABEL_EMB).to(device)
netD = Discriminator(num_gan_classes, GAN_LABEL_EMB, GAN_IMG_SIZE).to(device)
netG.apply(weights_init)
netD.apply(weights_init)

optD = torch.optim.Adam(netD.parameters(), lr=GAN_LR, betas=(0.5, 0.999))
optG = torch.optim.Adam(netG.parameters(), lr=GAN_LR, betas=(0.5, 0.999))
gan_criterion = nn.BCELoss()

print(f'Generator:     {sum(p.numel() for p in netG.parameters()):,} params')
print(f'Discriminator: {sum(p.numel() for p in netD.parameters()):,} params')
print(f'Training for {GAN_EPOCHS} epochs ...')
print('Healthy: D_loss ~1.0-1.4, G_loss ~1.5-3.0')

REAL_LABEL = 0.9
FAKE_LABEL = 0.0

for epoch in range(GAN_EPOCHS):
    d_total, g_total, n_batches = 0.0, 0.0, 0
    for real_imgs, labels in gan_loader:
        real_imgs, labels = real_imgs.to(device), labels.to(device)
        b = real_imgs.size(0)
        netD.zero_grad()
        d_real = gan_criterion(netD(real_imgs, labels), torch.full((b,), REAL_LABEL, device=device))
        d_real.backward()
        fake = netG(torch.randn(b, GAN_NOISE_DIM, device=device), labels)
        d_fake = gan_criterion(netD(fake.detach(), labels), torch.full((b,), FAKE_LABEL, device=device))
        d_fake.backward()
        optD.step()
        netG.zero_grad()
        g_loss = gan_criterion(netD(fake, labels), torch.full((b,), REAL_LABEL, device=device))
        g_loss.backward()
        optG.step()
        d_total += d_real.item() + d_fake.item()
        g_total += g_loss.item()
        n_batches += 1
    if (epoch + 1) % 25 == 0 or epoch == 0:
        print(f'  Epoch [{epoch+1:3d}/{GAN_EPOCHS}]  D_loss: {d_total/n_batches:.4f}  G_loss: {g_total/n_batches:.4f}')

torch.save({'netG': netG.state_dict(), 'gan_class_list': gan_class_list,
            'gan_label_to_idx': gan_label_to_idx},
           '/content/drive/MyDrive/dcgan_petridish_dino.pth')
print('GAN training complete. Model saved.')

## STEP 8 — Generate synthetic images & augment training dataframe

In [ ]:
print('Generating synthetic images ...')
gan_gen_dir = '/content/gan_generated'
os.makedirs(gan_gen_dir, exist_ok=True)

def tensor_to_pil(t):
    arr = ((t.clamp(-1, 1) + 1) * 127.5).byte().cpu().permute(1, 2, 0).numpy()
    return Image.fromarray(arr)

netG.eval()
generated_rows = []
with torch.no_grad():
    for cls in gan_class_list:
        existing = class_counts[cls]
        to_gen   = max(0, GAN_TARGET - existing)
        if to_gen == 0:
            continue
        cls_dir = os.path.join(gan_gen_dir, cls)
        os.makedirs(cls_dir, exist_ok=True)
        cls_idx   = gan_label_to_idx[cls]
        generated = 0
        while generated < to_gen:
            batch = min(GAN_BATCH_SIZE, to_gen - generated)
            noise = torch.randn(batch, GAN_NOISE_DIM, device=device)
            lbls  = torch.full((batch,), cls_idx, dtype=torch.long, device=device)
            fakes = netG(noise, lbls)
            for i, img_t in enumerate(fakes):
                fpath = os.path.join(cls_dir, f'gan_{cls}_{generated+i:05d}.png')
                tensor_to_pil(img_t).save(fpath)
                generated_rows.append({'filepath': fpath, 'label': cls})
            generated += batch
        print(f'  [{cls}] {existing} real + {to_gen} synthetic = {existing+to_gen} total')

gen_df = pd.DataFrame(generated_rows)
gen_df['label_idx'] = gen_df['label'].map(label_to_idx)

train_split_df_aug = (
    pd.concat([train_split_df, gen_df], ignore_index=True)
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)
print(f'Original train:  {len(train_split_df)}')
print(f'Augmented train: {len(train_split_df_aug)}')
print('\nPost-GAN class distribution:')
print(train_split_df_aug['label'].value_counts().sort_index())

## STEP 9 — DINOv2 transforms, Dataset, DataLoader & Focal Loss

**Focal Loss** focuses on hard/rare examples: $FL = -(1-p_t)^\gamma \log(p_t)$
- Confident correct predictions → $(1-p_t)^\gamma$ small → low weight
- Wrong/uncertain predictions → $(1-p_t)^\gamma$ large → high weight
- γ=2 focuses 5× more on hard examples vs standard CE

In [ ]:
DINOV2_MEAN = [0.485, 0.456, 0.406]
DINOV2_STD  = [0.229, 0.224, 0.225]
IMG_SIZE    = 448   # 32x14px patches — preserves cell morphology detail

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(45),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.15),
    transforms.RandomApply([transforms.GaussianBlur(5)], p=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=DINOV2_MEAN, std=DINOV2_STD),
])

val_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE + 32),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=DINOV2_MEAN, std=DINOV2_STD),
])


class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None, label_smoothing=0.0):
        super().__init__()
        self.gamma = gamma
        self.label_smoothing = label_smoothing
        if weight is not None:
            self.register_buffer('weight', weight)
        else:
            self.weight = None

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.weight, reduction='none')
        pt = torch.exp(-ce)
        ce_s = (1 - self.label_smoothing) * ce + self.label_smoothing * (-torch.log(pt + 1e-8)).mean()
        return ((1 - pt) ** self.gamma * ce_s).mean()


class ImageDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return self.transform(Image.open(row['filepath']).convert('RGB')), int(row['label_idx'])


BATCH_SIZE = 8   # DINOv2-B at 448px — reduce to 4 if OOM

train_dataset = ImageDataset(train_split_df_aug, train_transform)
val_dataset   = ImageDataset(val_split_df,       val_transform)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

counts = train_split_df_aug['label_idx'].value_counts().sort_index()
fw = 1.0 / torch.tensor(counts.values, dtype=torch.float).sqrt()
fw = (fw / fw.sum() * num_classes).to(device)
criterion = FocalLoss(gamma=2.0, weight=fw, label_smoothing=0.1)

print(f'Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}')

## STEP 10 — Load DINOv2-ViT-B/14 + classifier head

In [ ]:
print('Loading DINOv2-ViT-B/14 from torch.hub ...')
backbone = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14', pretrained=True)


class DINOv2Classifier(nn.Module):
    """
    DINOv2-ViT-B/14 with a classification head.
    CLS token (768-dim) → LayerNorm → Dropout → Linear(num_classes).
    """
    def __init__(self, backbone, num_classes, dropout=0.5):
        super().__init__()
        self.backbone = backbone
        embed_dim = backbone.embed_dim   # 768 for ViT-B
        self.head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Dropout(dropout),
            nn.Linear(embed_dim, num_classes),
        )

    def forward(self, x):
        cls_token = self.backbone(x)   # [B, 768]
        return self.head(cls_token)


model = DINOv2Classifier(backbone, num_classes=num_classes).to(device)
print(f'Backbone embed_dim: {backbone.embed_dim}')
print(f'Transformer blocks: {len(backbone.blocks)}')
print(f'Total params:       {sum(p.numel() for p in model.parameters()):,}')
best_model_path = '/content/drive/MyDrive/dcgan_dinov2_best.pth'

## STEP 11 — Phase 1: Train classifier head only (5 epochs)

Backbone frozen. Initialises head weights before any backbone updates. Mixed-precision (AMP) used throughout to fit DINOv2 in Colab VRAM.

In [ ]:
for param in model.backbone.parameters():
    param.requires_grad = False
for param in model.head.parameters():
    param.requires_grad = True

optimizer = torch.optim.AdamW(model.head.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=5)
scaler    = torch.amp.GradScaler('cuda')

num_epochs   = 5
best_val_acc = 0.0
print(f'Phase 1 trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

for epoch in range(num_epochs):
    model.train()
    run_loss, correct, total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            outputs = model(images)
            loss    = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        run_loss += loss.item()
        correct  += (outputs.argmax(1) == labels).sum().item()
        total    += labels.size(0)
    scheduler.step()

    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            with torch.amp.autocast('cuda'):
                outputs = model(images)
            val_correct += (outputs.argmax(1) == labels).sum().item()
            val_total   += labels.size(0)
    val_acc = val_correct / val_total

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({'model_state_dict': model.state_dict(), 'label_to_idx': label_to_idx,
                    'idx_to_label': idx_to_label, 'best_val_acc': best_val_acc}, best_model_path)
        print(f'  --> Saved (val_acc={val_acc:.4f})')
    print(f'Phase 1 | Ep {epoch+1}/{num_epochs} | Loss: {run_loss/len(train_loader):.4f} | '
          f'Train: {correct/total:.4f} | Val: {val_acc:.4f} | LR: {scheduler.get_last_lr()[0]:.2e}')
print(f'Phase 1 best val acc: {best_val_acc:.4f}')

## STEP 12 — Phase 2: Unfreeze last 4 transformer blocks + head (20 epochs)

DINOv2-ViT-B has 12 blocks (0–11). Blocks 8–11 capture high-level visual semantics. Adapting these to cell morphology while keeping early blocks frozen prevents catastrophic forgetting.

In [ ]:
checkpoint = torch.load(best_model_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
print(f'Starting Phase 2 from best val acc: {checkpoint["best_val_acc"]:.4f}')

for param in model.backbone.parameters():
    param.requires_grad = False
for block in model.backbone.blocks[-4:]:
    for param in block.parameters():
        param.requires_grad = True
for param in model.backbone.norm.parameters():
    param.requires_grad = True
for param in model.head.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Phase 2 trainable params: {trainable:,}')

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()), lr=2e-5, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)
scaler    = torch.amp.GradScaler('cuda')

num_epochs = 20
for epoch in range(num_epochs):
    model.train()
    run_loss, correct, total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            outputs = model(images)
            loss    = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        run_loss += loss.item()
        correct  += (outputs.argmax(1) == labels).sum().item()
        total    += labels.size(0)

    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            with torch.amp.autocast('cuda'):
                outputs = model(images)
            val_correct += (outputs.argmax(1) == labels).sum().item()
            val_total   += labels.size(0)
    val_acc = val_correct / val_total
    scheduler.step(val_acc)
    current_lr = optimizer.param_groups[0]['lr']

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({'model_state_dict': model.state_dict(), 'label_to_idx': label_to_idx,
                    'idx_to_label': idx_to_label, 'best_val_acc': best_val_acc}, best_model_path)
        print(f'  --> Saved at epoch {epoch+1} (val_acc={val_acc:.4f})')
    print(f'Phase 2 | Ep {epoch+1}/{num_epochs} | Loss: {run_loss/len(train_loader):.4f} | '
          f'Train: {correct/total:.4f} | Val: {val_acc:.4f} | LR: {current_lr:.2e}')
print(f'Phase 2 best val acc: {best_val_acc:.4f}')

## STEP 13 — Phase 3: Full model fine-tune (15 epochs)

Loads best Phase 2 checkpoint. All layers unfrozen at lr=5e-6. Higher weight_decay (5e-4) to combat overfitting.

In [ ]:
checkpoint = torch.load(best_model_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device)
print(f'Starting Phase 3 from best val acc: {checkpoint["best_val_acc"]:.4f}')

for param in model.parameters():
    param.requires_grad = True
print(f'Phase 3 trainable params: {sum(p.numel() for p in model.parameters()):,}')

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-6, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)
scaler    = torch.amp.GradScaler('cuda')

num_epochs = 15
for epoch in range(num_epochs):
    model.train()
    run_loss, correct, total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            outputs = model(images)
            loss    = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        run_loss += loss.item()
        correct  += (outputs.argmax(1) == labels).sum().item()
        total    += labels.size(0)

    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            with torch.amp.autocast('cuda'):
                outputs = model(images)
            val_correct += (outputs.argmax(1) == labels).sum().item()
            val_total   += labels.size(0)
    val_acc = val_correct / val_total
    scheduler.step(val_acc)
    current_lr = optimizer.param_groups[0]['lr']

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({'model_state_dict': model.state_dict(), 'label_to_idx': label_to_idx,
                    'idx_to_label': idx_to_label, 'best_val_acc': best_val_acc}, best_model_path)
        print(f'  --> Saved at phase3 epoch {epoch+1} (val_acc={val_acc:.4f})')
    print(f'Phase 3 | Ep {epoch+1}/{num_epochs} | Loss: {run_loss/len(train_loader):.4f} | '
          f'Train: {correct/total:.4f} | Val: {val_acc:.4f} | LR: {current_lr:.2e}')
print(f'Best val acc after Phase 3: {best_val_acc:.4f}')
print(f'Best model saved to: {best_model_path}')

## STEP 14 — Evaluate best model on validation set

In [ ]:
checkpoint = torch.load(best_model_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f'Evaluating best model (val_acc={checkpoint["best_val_acc"]:.4f})')

all_true, all_pred = [], []
with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        with torch.amp.autocast('cuda'):
            outputs = model(images)
        all_true.extend(labels.numpy())
        all_pred.extend(outputs.argmax(1).cpu().numpy())

class_names = [idx_to_label[i] for i in range(num_classes)]
cm = confusion_matrix(all_true, all_pred)

plt.figure(figsize=(12, 10))
plt.imshow(cm, interpolation='nearest')
plt.title('Validation Confusion Matrix (DC-GAN + DINOv2-ViT-B/14)')
plt.colorbar()
tick_marks = np.arange(num_classes)
plt.xticks(tick_marks, class_names, rotation=45, ha='right')
plt.yticks(tick_marks, class_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.show()

print(classification_report(all_true, all_pred, target_names=class_names, zero_division=0))

## STEP 15 — Build test DataLoader & generate submission CSV

In [ ]:
class TestDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return self.transform(Image.open(row['filepath']).convert('RGB')), row['filename']

test_rows = [{'filepath': os.path.join(test_dir, f), 'filename': f}
             for f in sorted(os.listdir(test_dir)) if f.lower().endswith(image_exts)]
test_df      = pd.DataFrame(test_rows)
test_dataset = TestDataset(test_df, val_transform)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f'Test images: {len(test_df)}  |  Batches: {len(test_loader)}')

model.eval()
all_filenames, all_preds = [], []
with torch.no_grad():
    for images, filenames in test_loader:
        images = images.to(device)
        with torch.amp.autocast('cuda'):
            outputs = model(images)
        all_filenames.extend(filenames)
        all_preds.extend([idx_to_label[p.item()] for p in outputs.argmax(1).cpu()])

submission_df = pd.DataFrame({'TestFileName': all_filenames, 'Class': all_preds})
csv_path = '/content/drive/MyDrive/dcgan_dinov2_submission.csv'
submission_df.to_csv(csv_path, index=False)

print(submission_df.head(10))
print(f'\nPrediction distribution:\n{submission_df["Class"].value_counts().sort_index()}')
print(f'\nSaved to: {csv_path}')